# Sistema de detección de enlaces spam

## 1. Carga de datos

In [17]:
import pandas as pd
import numpy as np

import seaborn as sns
import matplotlib.pyplot as plt

import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.model_selection import GridSearchCV

import pickle
import os

In [18]:
df = pd.read_csv('/workspaces/IgnacioSabinoG-IntroML/data/raw/url_spam.csv')
df

,url,is_spam
0,https://briefingday.us8.list-manage.com/unsubs...,True
1,https://www.hvper.com/,True
2,https://briefingday.com/m/v4n3i4f3,True
3,https://briefingday.com/n/20200618/m#commentform,False
4,https://briefingday.com/fan,True
...,...,...
2994,https://www.smartcitiesworld.net/news/news/dee...,False
2995,https://www.youtube.com/watch,True
2996,https://techcrunch.com/2019/07/04/an-optimisti...,False
2997,https://www.technologyreview.com/2019/12/20/13...,False


## 2. Procesamiento de las URLs

In [19]:
# Descargar recursos necesarios de nltk
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

def preprocess_url(url):
    # 1. Eliminar protocolos (http, https) y "www"
    url = re.sub(r'https?://(www\.)?', '', url)
    # 2. Reemplazar signos de puntuación por espacios para segmentar la URL
    url = re.sub(r'[^\w\s]', ' ', url)
    # 3. Convertir a minúsculas y separar por palabras
    words = url.lower().split()
    # 4. Eliminar stopwords y lematizar
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    
    return " ".join(words)

# Aplicar la limpieza a la columna 'url'
df['clean_url'] = df['url'].apply(preprocess_url)

# Convertir la columna objetivo (is_spam) a valores numéricos (0 y 1)
df['is_spam'] = df['is_spam'].astype(int)

[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/vscode/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [20]:
df

,url,is_spam,clean_url
0,https://briefingday.us8.list-manage.com/unsubs...,1,briefingday us8 list manage com unsubscribe
1,https://www.hvper.com/,1,hvper com
2,https://briefingday.com/m/v4n3i4f3,1,briefingday com v4n3i4f3
3,https://briefingday.com/n/20200618/m#commentform,0,briefingday com n 20200618 commentform
4,https://briefingday.com/fan,1,briefingday com fan
...,...,...,...
2994,https://www.smartcitiesworld.net/news/news/dee...,0,smartcitiesworld net news news deepfake techno...
2995,https://www.youtube.com/watch,1,youtube com watch
2996,https://techcrunch.com/2019/07/04/an-optimisti...,0,techcrunch com 2019 07 04 optimistic view deep...
2997,https://www.technologyreview.com/2019/12/20/13...,0,technologyreview com 2019 12 20 131462 startup...


## 3. División en Train y Test


In [21]:
X = df['clean_url']
y = df['is_spam']

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorización (Transformar texto a números)
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

print(f"Forma de X_train: {X_train.shape}")

Forma de X_train: (2399, 5446)


## 4. Construcción del SVM (Modelo Base)


In [22]:
# Inicializar y entrenar el modelo
svm_model = SVC(kernel='linear') # El kernel lineal suele funcionar muy bien con texto
svm_model.fit(X_train, y_train)

# Predicciones
y_pred = svm_model.predict(X_test)

# Resultados
print("--- Reporte de Clasificación (Modelo Base) ---")
print(classification_report(y_test, y_pred))

--- Reporte de Clasificación (Modelo Base) ---
              precision    recall  f1-score   support

           0       0.96      0.97      0.97       455
           1       0.91      0.88      0.89       145

    accuracy                           0.95       600
   macro avg       0.93      0.92      0.93       600
weighted avg       0.95      0.95      0.95       600



Una precisión (accuracy) del 95% indica que el SVM está identificando muy bien los patrones en las URLs.

Clase 0 (No Spam): Tienes un F1-score de 0.97, lo cual es casi perfecto.

Clase 1 (Spam): El recall es de 0.88, lo que significa que el modelo detecta el 88% del spam real.

Hay un margen de mejora ya que un 12% se está "escapando" y lo que nos interesa es realmente identificar el spam.

## 5. Grid Search

In [23]:
# Definimos los parámetros a probar
param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto']
}

# Inicializamos el GridSearch con validación cruzada (cv=5)
grid = GridSearchCV(SVC(), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

# Mejores parámetros encontrados
print(f"Mejores parámetros: {grid.best_params_}")

# Evaluamos el nuevo modelo optimizado
best_model = grid.best_estimator_
y_pred_optimized = best_model.predict(X_test)

print("\n--- Reporte de Clasificación (Modelo Optimizado) ---")
print(classification_report(y_test, y_pred_optimized))

Mejores parámetros: {'C': 10, 'gamma': 'scale', 'kernel': 'rbf'}

--- Reporte de Clasificación (Modelo Optimizado) ---
              precision    recall  f1-score   support

           0       0.98      0.96      0.97       455
           1       0.88      0.93      0.90       145

    accuracy                           0.95       600
   macro avg       0.93      0.94      0.94       600
weighted avg       0.95      0.95      0.95       600



Subió el Recall del Spam (Clase 1): Se pasó de 0.88 a 0.93, lo cual es clave en un filtro de spam; ahora el sistema es mucho más "sensible" y detecta el 93% de los enlaces maliciosos, dejando escapar muy pocos.

Ligero intercambio en Precisión: Bajó un poco la precisión de la clase 1 (de 0.91 a 0.88), lo que significa que quizás algún enlace legítimo sea marcado como spam, pero a cambio se tiene un sistema mucho más seguro.

F1-Score: El promedio armónico mejoró, lo que indica un modelo más equilibrado.

El uso de un kernel rbf con una C=10 permitió al SVM encontrar una frontera de decisión más compleja y flexible que la línea recta inicial.

## 6. Guardar modelo

In [24]:
# Crear carpeta si no existe
if not os.path.exists('models'):
    os.makedirs('models')

# Guardar el modelo optimizado
with open('models/svm_model_optimized.pkl', 'wb') as f:
    pickle.dump(best_model, f)

# IMPORTANTE: También guarda el vectorizador, sin él no podrás usar el modelo después
with open('models/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Modelo y vectorizador guardados en la carpeta /models")

Modelo y vectorizador guardados en la carpeta /models
